# First Look at the Soccer Data

Run `python scripts/load_data.py` first to populate the DB.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src.db import get_connection
conn = get_connection()
for table in ['competitions', 'teams', 'matches']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {table}', conn).iloc[0]['n']
    print(f'{table:15s} {n:>8,} rows')

competitions          75 rows
teams                312 rows
matches            3,464 rows


In [2]:
competitions = pd.read_sql_query('SELECT * FROM competitions ORDER BY competition_name, season_name', conn)
competitions.head(20)

,competition_id,season_id,country_name,competition_name,season_name
0,9,27,Germany,1. Bundesliga,2015/2016
1,9,281,Germany,1. Bundesliga,2023/2024
2,1267,107,Africa,African Cup of Nations,2023
3,16,276,Europe,Champions League,1970/1971
4,16,71,Europe,Champions League,1971/1972
5,16,277,Europe,Champions League,1972/1973
6,16,76,Europe,Champions League,1999/2000
7,16,44,Europe,Champions League,2003/2004
8,16,37,Europe,Champions League,2004/2005
9,16,39,Europe,Champions League,2006/2007


## Top scorers in a competition (default: World Cup 2018)

In [3]:
COMP_ID, SEASON_ID = 43, 3
sql = '''
WITH all_team_goals AS (
    SELECT home_team_id AS team_id, home_score AS goals FROM matches WHERE competition_id=? AND season_id=?
    UNION ALL
    SELECT away_team_id AS team_id, away_score AS goals FROM matches WHERE competition_id=? AND season_id=?
)
SELECT t.team_name, SUM(g.goals) AS total_goals, COUNT(*) AS matches_played
FROM all_team_goals g JOIN teams t ON t.team_id=g.team_id
GROUP BY t.team_name ORDER BY total_goals DESC LIMIT 10
'''
pd.read_sql_query(sql, conn, params=(COMP_ID, SEASON_ID, COMP_ID, SEASON_ID))

,team_name,total_goals,matches_played
0,Belgium,16,7
1,France,14,7
2,Croatia,14,7
3,England,12,7
4,Russia,11,5
5,Brazil,8,5
6,Uruguay,7,5
7,Spain,7,4
8,Sweden,6,5
9,Portugal,6,4


## Elo ratings

In [4]:
from src.elo import fit_ratings
matches_in_order = pd.read_sql_query(
    'SELECT * FROM matches WHERE competition_id=? AND season_id=? AND home_score IS NOT NULL ORDER BY match_date, kick_off',
    conn, params=(COMP_ID, SEASON_ID))
ratings = fit_ratings(matches_in_order.to_dict('records'))
teams_df = pd.read_sql_query('SELECT * FROM teams', conn)
(pd.DataFrame([{'team_id': k, 'rating': v} for k, v in ratings.items()])
    .merge(teams_df, on='team_id').sort_values('rating', ascending=False).head(15))

,team_id,rating,team_name
8,771,1544.470853,France
24,782,1543.987287,Belgium
14,785,1527.452967,Croatia
20,781,1518.892371,Brazil
3,783,1516.070055,Uruguay
27,768,1514.485410,England
28,769,1514.059174,Colombia
0,796,1510.974946,Russia
22,790,1508.578683,Sweden
13,776,1506.591010,Denmark


## Predict a match (Poisson)

In [5]:
from src.poisson import compute_team_strengths, expected_goals, match_outcome_probabilities
matches_records = matches_in_order.to_dict('records')
strengths, lh, la = compute_team_strengths(matches_records)
HOME_TEAM_ID, AWAY_TEAM_ID = 771, 785  # France vs Croatia (WC18 final)
lam_h, lam_a = expected_goals(HOME_TEAM_ID, AWAY_TEAM_ID, strengths, lh, la)
result = match_outcome_probabilities(lam_h, lam_a)
print(f'Expected goals: {lam_h:.2f} - {lam_a:.2f}')
print(f'Home win: {result["home_win"]:.1%} | Draw: {result["draw"]:.1%} | Away win: {result["away_win"]:.1%}')

Expected goals: 2.95 - 2.22
Home win: 53.5% | Draw: 17.2% | Away win: 28.9%


## Season simulation (Monte Carlo)

In [6]:
from src.monte_carlo import run_simulations
midpoint = len(matches_records) // 2
completed, remaining = matches_records[:midpoint], matches_records[midpoint:]
s_partial, lh, la = compute_team_strengths(completed)
summary = run_simulations(completed, remaining, s_partial, lh, la, n_simulations=5000)
(pd.DataFrame(summary.values()).merge(teams_df, on='team_id')
    .sort_values('avg_position')[['team_name','avg_position','avg_points','p_champion']].head(15))

,team_name,avg_position,avg_points,p_champion
17,Croatia,1.1404,17.3060,0.8874
0,England,2.8356,13.4252,0.0726
15,Uruguay,3.5722,12.7976,0.0078
3,France,4.4774,12.2118,0.0180
14,Belgium,5.7970,11.0210,0.0142
26,Mexico,7.2258,9.5250,0.0000
13,Brazil,7.5106,9.2482,0.0000
28,Russia,9.4254,7.6920,0.0000
4,Spain,9.5204,7.9032,0.0000
5,Switzerland,10.5140,7.2640,0.0000


In [7]:
conn.close()